# Мини-пакеты, нормализация и dropout

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Объяснять мини-пакеты, нормализацию и dropout и проверять различия train/eval.

## По сколько объектов учиться

Мини-пакет собирает несколько объектов для одного шага. Средняя потеря пакета делает масштаб градиента менее зависимым от числа объектов, чем сумма. Последний пакет может быть короче; его форму нужно брать из фактических данных. Перемешивают пары «признаки, ответ» вместе.

**Нормализация** приводит промежуточные значения к более удобному масштабу. BatchNorm использует статистики пакета при обучении и накопленные статистики при оценке. Это отличается от обычного предварительного StandardScaler: статистики находятся внутри модели и зависят от режима. Очень маленькие пакеты могут делать их ненадёжными.

Проследим нормализацию одного признака в пакете [2,4]. Среднее равно 3, средний квадрат отклонения — 1. Вычитаем 3 и делим на $\sqrt{1+\varepsilon}$, где маленькое $\varepsilon>0$ предотвращает деление на ноль: получаем приблизительно [−1,1]. BatchNorm затем применяет обучаемые масштаб и сдвиг. В eval вместо нового среднего пакета использует накопленные при обучении значения: один и тот же объект не должен менять ответ лишь от состава проверочного пакета.

## Разобранный пример: случайное выключение

Dropout при обучении случайно обнуляет часть активаций с вероятностью p. В рассматриваемом правиле 0≤p<1. Сохранившиеся значения умножаются на 1/(1−p), чтобы математическое ожидание сохранялось. При p=0.5 вход 2 превращается в 0 или 4 с равными шансами, среднее 2. В режиме eval dropout пропускает 2 без случайного выключения. При p=1 знаменатель этой формулы нулевой: формулу масштабирования нельзя применять к такому случаю.

Это регуляризация: сеть не должна полностью полагаться на одну активацию. Она не создаёт дополнительные независимые примеры и не гарантирует улучшение любой сети. Вероятность выключения выбирают по внутренней проверке.

## Два переключателя

model.eval() меняет поведение dropout и BatchNorm. torch.no_grad() отключает запись графа, но не переключает эти слои в режим оценки. Для обычного итогового прогноза нужны оба действия. Не сравнивайте качество случайно работающего dropout с детерминированной моделью, забыв о режиме.

В ноутбуке nn.Dropout демонстрируется на множестве одинаковых активаций. Мы проверяем допустимые значения и точное поведение eval, не требуя, чтобы случайная доля обнулений была ровно 50%.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
layer = nn.Dropout(p=0.5)
x = torch.full((100,), 2.)
layer.train()
out = layer(x)
assert set(out.tolist()).issubset({0., 4.})
layer.eval()
with torch.no_grad():
    stable = layer(x)
assert torch.equal(stable, x)
print("Нулей в учебном train-выходе:", int((out == 0).sum()))


## Типичные ошибки

Не путайте вероятность выключения и долю оставленных. Не забывайте eval и не требуйте точного совпадения случайной доли с ожиданием.

## Задачи

### 1. Базовая

Обязан ли последний пакет быть полным?

<details>
<summary>Подсказка</summary>

Размер данных может не делиться.

</details>



### 2. Базовая

Отключает ли no_grad случайность dropout?

<details>
<summary>Подсказка</summary>

Он управляет графом.

</details>



### 3. Прикладная

При p=0.5 сохранилось значение 3. Во что оно превращается при train?

<details>
<summary>Подсказка</summary>

Разделите на 1−p.

</details>



### 4. Прикладная

Пять объектов, batch_size=2. Назовите размеры пакетов без отбрасывания.

<details>
<summary>Подсказка</summary>

Последний короче.

</details>



### 5. Олимпиадная
Dropout независимо отключает каждое из четырёх ненулевых значений с вероятностью 0.5. Найдите вероятность, что обнулится ровно половина значений. Можно ли требовать этого в каждом тестовом запуске?

<details>
<summary>Подсказка</summary>

Выберите два отключённых места среди четырёх; каждый конкретный рисунок маски имеет одинаковую вероятность.

</details>



### 6. Олимпиадная

Почему BatchNorm в eval нельзя заново обучать по тестовым пакетам?

<details>
<summary>Подсказка</summary>

Меняются выученные статистики.

</details>



## Итоги и самопроверка

Вычислите ожидаемый выход одного dropout-элемента. Чем статистики BatchNorm при train отличаются от eval?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Нет. Это часть, которую нужно обрабатывать без жёсткой формы.

### Задача 2

Нет. Для режима без dropout нужен eval.

### Задача 3

В 6.

### Задача 4

2,2,1.

### Задача 5

Две нулевые позиции можно выбрать шестью способами: (1,2), (1,3), (1,4), (2,3), (2,4), (3,4). Всего 2⁴=16 равновероятных масок, поэтому вероятность равна 6/16=3/8. В остальных 5/8 запусков нулей будет другое число. Проверка «всегда ровно половина» неверна: вероятность отключения каждого значения не задаёт точную долю в маленьком пакете.

### Задача 6

Это адаптация к проверочным данным и другая процедура, которую обычный независимый тест не предполагает.